E-8: Completare la gestione della maschera e verificare con due test:
1) passare due volte lo stesso batch con model.eval(), le uscite devono
essere identiche bit per bit;
2) cambiare i valori nelle posizioni mascherate e verificare che l'uscita non cambi.

In [ ]:
import torch
import torch.nn as nn

print("PyTorch:", torch.__version__)
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

PyTorch: 2.11.0+cpu
Device: cpu


In [ ]:
class MiniNetEncoder(nn.Module):

    def __init__(self, n_pkt=20, d=64, layers=4, heads=4):
        super().__init__()

        self.inp = nn.Linear(3, d)

        self.cls = nn.Parameter(
            torch.zeros(1, 1, d)
        )

        self.pos = nn.Parameter(
            torch.zeros(1, n_pkt + 1, d)
        )

        enc = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=heads,
            dim_feedforward=4 * d,
            batch_first=True,
            norm_first=True
        )

        self.body = nn.TransformerEncoder(
            enc,
            num_layers=layers
        )

    def forward(self, x, pad_mask):

        # x: (B, n_pkt, 3)
        h = self.inp(x)

        # aggiungo il token CLS
        cls = self.cls.expand(h.size(0), -1, -1)

        h = torch.cat([cls, h], dim=1)

        # aggiungo positional embedding
        h = h + self.pos

        # pad_mask originale: (B, n_pkt)
        # il CLS non deve mai essere mascherato
        cls_mask = torch.zeros(
            (pad_mask.size(0), 1),
            dtype=torch.bool,
            device=pad_mask.device
        )

        pad_mask = torch.cat(
            [cls_mask, pad_mask],
            dim=1
        )

        return self.body(
            h,
            src_key_padding_mask=pad_mask
        )

In [ ]:
model = MiniNetEncoder()

print(model)

MiniNetEncoder(
  (inp): Linear(in_features=3, out_features=64, bias=True)
  (body): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
)


/tmp/ipykernel_227/2933949254.py:24: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.body = nn.TransformerEncoder(


In [ ]:
# Test del forward e della maschera

model.eval()

B = 4
n_pkt = 20

# Batch casuale: (B, 20, 3)
x = torch.randn(B, n_pkt, 3)

# Prime posizioni valide, ultime mascherate
pad_mask = torch.zeros(B, n_pkt, dtype=torch.bool)

pad_mask[:, 10:] = True

with torch.inference_mode():
    out = model(x, pad_mask)

print("Input:", x.shape)
print("Pad mask:", pad_mask.shape)
print("Output:", out.shape)

Input: torch.Size([4, 20, 3])
Pad mask: torch.Size([4, 20])
Output: torch.Size([4, 21, 64])


In [ ]:
# E-8 — Test 1: determinismo

model.eval()

with torch.inference_mode():
    out1 = model(x, pad_mask)
    out2 = model(x, pad_mask)

identiche = torch.equal(out1, out2)

print("Uscite identiche bit per bit:", identiche)

if identiche:
    print("✓ Test di determinismo SUPERATO")
else:
    print("✗ Test di determinismo FALLITO")

Uscite identiche bit per bit: True
✓ Test di determinismo SUPERATO


In [ ]:
# E-8 — Test 2: verifica della maschera

model.eval()

# Copia dell'input
x_mod = x.clone()

# Modifichiamo SOLO le posizioni mascherate
x_mod[pad_mask] = torch.randn_like(x_mod[pad_mask])

with torch.inference_mode():
    out_originale = model(x, pad_mask)
    out_modificato = model(x_mod, pad_mask)

# Verifichiamo CLS
cls_identico = torch.equal(
    out_originale[:, 0, :],
    out_modificato[:, 0, :]
)

# Verifichiamo le posizioni valide
valid_identico = torch.equal(
    out_originale[:, 1:11, :],
    out_modificato[:, 1:11, :]
)

print("CLS identico:", cls_identico)
print("Posizioni valide identiche:", valid_identico)

if cls_identico and valid_identico:
    print("✓ Test della maschera SUPERATO")
else:
    print("✗ Test della maschera FALLITO")

CLS identico: True
Posizioni valide identiche: True
✓ Test della maschera SUPERATO
